In [190]:
import pandas as pd
import numpy as np

In [191]:
clean_inventario_demanda_df = pd.read_parquet('../data/processed/inventario_demanda_limpio.parquet')
clean_inventario_demanda_df

,fecha,id_producto,nombre_producto,categoria,id_sucursal,nombre_sucursal,proveedor,unidades_vendidas,unidades_recibidas,stock_cierre,precio_unitario,costo_unitario,importe_venta,importe_costo
0,2024-06-06,P025,Papel Higiénico 4-pack,Higiene,S01,Mérida Centro,Grupo Abasto del Sur,45,0,56,61.01,30.0,2745.45,1350.0
1,2023-08-17,P007,Jabón Multiusos 500ml,Limpieza,S05,Playa del Carmen,Proveedor Central del Caribe,9,0,146,41.74,19.0,375.66,171.0
2,2023-12-14,P018,Café Molido 250g,Abarrotes,S04,Cancún Centro,Comercializadora Peninsular,14,0,203,84.89,42.0,1188.46,588.0
3,2023-07-09,P006,Detergente Líquido 1L,Limpieza,S04,Cancún Centro,Logística Maya S.A.,16,0,218,72.62,35.0,1161.92,560.0
4,2023-11-25,P010,Esponja Cocina Pack 3,Limpieza,S01,Mérida Centro,Comercializadora Peninsular,13,95,278,33.61,15.0,436.93,195.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52734,2023-11-17,P006,Detergente Líquido 1L,Limpieza,S02,Mérida Norte,N/A,8,0,142,75.26,35.0,602.08,280.0
52735,2024-06-28,P025,Papel Higiénico 4-pack,Higiene,S03,Cancún,Distribuidora Norte S.A.,23,175,175,63.07,30.0,1450.61,690.0
52736,2023-12-04,P021,Queso Manchego 400g,Lácteos,S02,Mérida Norte,Grupo Abasto del Sur,8,188,188,92.44,50.0,739.52,400.0
52737,2024-03-02,P001,Agua Mineral 1L,Bebidas,S02,Mérida Norte,Comercializadora Peninsular,23,0,131,18.69,8.0,429.87,184.0


In [192]:
semanal_df = (
    clean_inventario_demanda_df
    .groupby(
        [pd.Grouper(key="fecha", freq="W-MON", label="left", closed="left"), "nombre_producto", "nombre_sucursal"], observed=True,
    )["unidades_vendidas"].sum()
    .unstack(["nombre_producto", "nombre_sucursal"], fill_value=0)
    .asfreq("W-MON", fill_value=0) 
)

semanal_df

nombre_producto Aceite Vegetal 1L                                           \
nombre_sucursal            Cancún Cancún Centro Mérida Centro Mérida Norte   
fecha                                                                        
2022-12-26                     26            19            18            9   
2023-01-02                     78            49            79           96   
2023-01-09                     57            75            62          142   
2023-01-16                     95           103            61           46   
2023-01-23                     78            63            61           77   
...                           ...           ...           ...          ...   
2024-05-27                     53            40            53           80   
2024-06-03                     65            77            66           54   
2024-06-10                     74            43            32           92   
2024-06-17                     51            58            31           39   
2024-06-24                     42            62            49           75   

nombre_producto Agua Mineral 1L                                           \
nombre_sucursal          Cancún Cancún Centro Mérida Centro Mérida Norte   
fecha                                                                      
2022-12-26                   66            65            56           55   
2023-01-02                  239           243           275          241   
2023-01-09                  232           304            96          354   
2023-01-16                  201           225           274          232   
2023-01-23                  318           300           176          271   
...                         ...           ...           ...          ...   
2024-05-27                  222           241           216          339   
2024-06-03                  285           282           289          265   
2024-06-10                  324           229           287          307   
2024-06-17                  277           294           347          208   
2024-06-24                  245           265           287          268   

nombre_producto                  Agua Sabor Limón 600ml  ...  Sal de Mesa 1kg  \
nombre_sucursal Playa del Carmen          Cancún Centro  ... Playa del Carmen   
fecha                                                    ...                    
2022-12-26                    66                     16  ...                0   
2023-01-02                   294                     80  ...               13   
2023-01-09                   318                     55  ...               14   
2023-01-16                   221                     92  ...               10   
2023-01-23                   218                     68  ...               11   
...                          ...                    ...  ...              ...   
2024-05-27                   306                     44  ...               19   
2024-06-03                   101                    105  ...                2   
2024-06-10                   324                     93  ...               10   
2024-06-17                   208                    105  ...                3   
2024-06-24                   257                     76  ...               13   

nombre_producto Shampoo 400ml Suavizante Ropa 1L Toallas Húmedas 40pz  \
nombre_sucursal Mérida Centro       Mérida Norte         Mérida Norte   
fecha                                                                   
2022-12-26                  0                  0                    0   
2023-01-02                 74                 19                   10   
2023-01-09                113                  6                   23   
2023-01-16                111                  9                   11   
2023-01-23                 75                  8                   11   
...                       ...                ...                  ...   
2024-05-27                 72                  7  

In [193]:
# Promedios
modelos = {
    "Promedio 1 semana": semanal_df.rolling(1).mean().shift(1),
    "Promedio 4 semanas": semanal_df.rolling(4).mean().shift(1),
    "Promedio 8 semanas": semanal_df.rolling(8).mean().shift(1),
    "Promedio 26 semanas": semanal_df.rolling(26).mean().shift(1),
    "Promedio 52 semanas": semanal_df.rolling(52).mean().shift(1),
}

modelos["Mezcla 1 y 52"] = (modelos["Promedio 1 semana"] + modelos["Promedio 52 semanas"]) / 2
modelos["Mezcla 4 y 52"] = (modelos["Promedio 4 semanas"] + modelos["Promedio 52 semanas"]) / 2
modelos["Mezcla 8 y 52"] = (modelos["Promedio 8 semanas"] + modelos["Promedio 52 semanas"]) / 2
modelos["Mezcla 1, 4, y 52"] = (modelos["Promedio 1 semana"] + modelos["Promedio 4 semanas"] + modelos["Promedio 52 semanas"]) / 3

In [194]:
# Multiplicador: Mes por baseline de producto
semanal_prod = semanal_df.T.groupby(level="nombre_producto", observed=True).sum().T

ventas_2023 = semanal_prod.loc["2023"]
prom_semanal_mes = ventas_2023.groupby(ventas_2023.index.month).mean()
indice = prom_semanal_mes / prom_semanal_mes.mean()

indice

nombre_producto,Aceite Vegetal 1L,Agua Mineral 1L,Agua Sabor Limón 600ml,Arroz Largo 1kg,Atún Lata 140g,Azúcar Blanca 1kg,Café Molido 250g,Cerveza Lata 355ml,Cloro Blanqueador 1L,Crema Ácida 200g,...,Papel Higiénico 4-pack,Pasta Dental 100ml,Queso Manchego 400g,Refresco Cola 600ml,Sal de Mesa 1kg,Shampoo 400ml,Sopa Pasta 200g,Suavizante Ropa 1L,Toallas Húmedas 40pz,Yogurt Natural 1kg
fecha,,,,,,,,,,,,,,,,,,,,,
1,1.203293,1.030825,1.011959,1.115602,1.080287,1.100781,1.165109,1.056714,1.157082,1.279108,...,1.019337,1.145211,1.076385,1.086186,1.168142,1.050752,1.063833,1.048193,1.095732,1.103082
2,0.969283,0.887998,0.889108,0.967176,0.958120,0.973606,0.969019,0.937779,0.916969,1.038045,...,0.991475,0.931396,0.988501,0.902539,0.973451,0.944086,0.885608,0.973896,1.007958,1.001029
3,0.945995,0.802895,0.738772,0.989723,0.923818,0.934016,1.005755,0.824469,1.120214,1.111840,...,0.969366,1.037557,0.974894,0.757046,0.943953,0.962306,1.027373,1.124498,0.839161,0.869400
4,0.959647,0.853266,0.823811,0.939949,1.009254,0.984918,0.878403,0.775539,1.002048,0.959331,...,0.941676,0.955448,0.924468,0.795280,1.027532,0.905992,0.978137,1.280120,0.848806,1.125345
5,1.014415,0.985004,0.936639,0.917190,0.988928,1.030326,0.866648,0.960678,0.960454,0.940636,...,0.986366,1.045022,1.029642,0.956268,1.081613,1.007688,0.931406,0.855422,0.883530,1.106982
6,0.758081,1.193882,1.196614,0.897834,0.850313,0.749798,0.781256,1.029100,0.907515,0.831420,...,1.001993,0.836017,0.976495,1.047428,0.732547,0.987978,0.769734,1.074297,1.114058,0.892151
7,0.842883,1.158054,1.466915,0.828874,0.891433,0.798923,0.906486,1.296805,0.877265,0.712365,...,0.910637,0.962747,0.744057,1.268907,0.959685,0.903673,0.831067,0.767068,1.115023,0.850875
8,1.047179,1.147573,1.209521,1.037369,0.939158,0.993806,0.963305,1.210554,0.973688,1.028206,...,0.967005,1.078197,0.897255,1.306618,0.958702,1.041808,1.014003,0.978916,1.080299,0.872650
9,0.917085,0.846361,0.903534,0.929314,0.964512,0.917856,0.965754,0.906876,1.039861,0.880617,...,1.017877,0.956278,0.975694,0.921857,1.052114,1.005369,0.987051,1.059237,0.901857,0.917339


In [195]:
def modelo_mezcla_indice(alpha=1):
    idx = (
        (1 + alpha * (indice - 1))
        .reindex(columns=semanal_df.columns.get_level_values("nombre_producto"))
        .set_axis(semanal_df.columns, axis=1)
        .reindex(semanal_df.index.month)
        .set_axis(semanal_df.index)
    )
    base = (semanal_df / idx).replace([np.inf, -np.inf], np.nan)
    mezcla = (base.rolling(4).mean().shift(1) + base.rolling(52).mean().shift(1)) / 2
    return mezcla * idx

modelos["Mezcla 4 y 52 × índice"] = modelo_mezcla_indice()
modelos["Mezcla 4 y 52 × índice 0.5"] = modelo_mezcla_indice(alpha=0.5)

In [196]:
real = semanal_df.loc["2024"]

mascara = real.notna()
for pred in modelos.values():
    mascara &= pred.loc["2024"].notna()

def calcular_errores(pred, real, mascara):
    error = (pred.loc["2024"] - real)[mascara]
    error = error.stack(["nombre_producto", "nombre_sucursal"], future_stack=True).dropna()
    return {
        "MAE": error.abs().mean(),
        "Sesgo": error.mean(),
    }

resultados = pd.DataFrame(
    {nombre: calcular_errores(pred, real, mascara) for nombre, pred in modelos.items()}
).T.sort_values("MAE")

resultados

,MAE,Sesgo
Mezcla 4 y 52 × índice 0.5,20.134836,0.821740
Mezcla 4 y 52 × índice,20.212276,-0.191861
Mezcla 4 y 52,20.535944,2.353844
Mezcla 8 y 52,20.539816,2.900078
Promedio 52 semanas,20.773491,2.237751
Promedio 26 semanas,21.181272,4.961844
"Mezcla 1, 4, y 52",21.260866,1.937349
Promedio 8 semanas,21.585160,3.562404
Mezcla 1 y 52,21.725010,1.671055
Promedio 4 semanas,22.130962,2.469936
